# ogbn-proteins: GNN Training with DGL

This notebook trains a GNN on the **ogbn-proteins** benchmark from the Open Graph Benchmark (OGB)
using the **Deep Graph Library (DGL)**.

The model supports the following MPNN types:
- `gat`  – Graph Attention Network (no edge features)
- `gate` – GAT with edge features
- `sage` – GraphSAGE
- `gcn`  – Graph Convolutional Network

Reference: *Classic GNNs are Strong Baselines: Reassessing GNNs for Node Classification* (NeurIPS 2024)

## 1. Install Dependencies

Run the cell below **once** if the required packages are not yet installed.

In [ ]:
# Uncomment and run if packages are missing
# !pip install dgl -f https://data.dgl.ai/wheels/repo.html
# !pip install ogb
# !pip install torch

## 2. Imports

In [ ]:
import os
import random
import time

import dgl
import dgl.function as fn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from dgl.dataloading import DataLoader, MultiLayerNeighborSampler
from dgl.nn.pytorch.conv import GraphConv, SAGEConv
from dgl.ops import edge_softmax
from dgl.utils import expand_as_pair
from ogb.nodeproppred import DglNodePropPredDataset, Evaluator

print('All imports successful.')

## 3. Configuration

Edit the variables below to configure the experiment (replaces command-line arguments).

In [ ]:
# ── Device ──────────────────────────────────────────────────────────────────
USE_CPU  = False   # Set True to force CPU mode
GPU_ID   = 0       # GPU device ID (ignored when USE_CPU=True)

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED     = 0
N_RUNS   = 1       # Number of independent runs

# ── Model ────────────────────────────────────────────────────────────────────
MPNN       = 'gat' # 'gat' | 'gate' | 'sage' | 'gcn'
N_LAYERS   = 6
N_HEADS    = 6
N_HIDDEN   = 80
USE_LABELS = False # Concatenate training labels as input features
NO_ATTN_DST= False # Disable destination attention in GAT
JK         = False # Enable Jumping Knowledge (JK) aggregation

# ── Regularisation ───────────────────────────────────────────────────────────
DROPOUT    = 0.25
INPUT_DROP = 0.1
ATTN_DROP  = 0.0
EDGE_DROP  = 0.1

# ── Optimiser ────────────────────────────────────────────────────────────────
LR          = 0.01
WEIGHT_DECAY= 0.0

# ── Training schedule ────────────────────────────────────────────────────────
N_EPOCHS    = 1000
EVAL_EVERY  = 5
LOG_EVERY   = 5

# ── Misc ─────────────────────────────────────────────────────────────────────
SAVE_PRED   = False  # Save final predictions to ./output/

# ── Dataset constants (do not change) ────────────────────────────────────────
DATASET_NAME  = 'ogbn-proteins'
N_NODE_FEATS  = 0    # will be set after preprocessing
N_EDGE_FEATS  = 8
N_CLASSES     = 112

# ── Device setup ─────────────────────────────────────────────────────────────
if USE_CPU or not torch.cuda.is_available():
    device = torch.device('cpu')
else:
    device = torch.device(f'cuda:{GPU_ID}')

print(f'Using device: {device}')

## 4. Model Definition

Custom GATConv (supports edge features and edge-drop) and the GNN wrapper.

In [ ]:
class GATConv(nn.Module):
    """Graph Attention Convolution with optional edge features and edge-drop."""

    def __init__(
        self,
        node_feats,
        edge_feats,
        out_feats,
        n_heads=1,
        attn_drop=0.0,
        edge_drop=0.0,
        negative_slope=0.2,
        residual=True,
        activation=None,
        use_attn_dst=True,
        allow_zero_in_degree=True,
        use_symmetric_norm=False,
    ):
        super().__init__()
        self._n_heads = n_heads
        self._in_src_feats, self._in_dst_feats = expand_as_pair(node_feats)
        self._out_feats = out_feats
        self._allow_zero_in_degree = allow_zero_in_degree
        self._use_symmetric_norm = use_symmetric_norm

        # Feature projections
        self.src_fc = nn.Linear(self._in_src_feats, out_feats * n_heads, bias=False)
        if residual:
            self.dst_fc = nn.Linear(self._in_src_feats, out_feats * n_heads)
            self.bias = None
        else:
            self.dst_fc = None
            self.bias = nn.Parameter(torch.zeros(out_feats * n_heads))

        # Attention projections
        self.attn_src_fc = nn.Linear(self._in_src_feats, n_heads, bias=False)
        self.attn_dst_fc = nn.Linear(self._in_src_feats, n_heads, bias=False) if use_attn_dst else None
        self.attn_edge_fc = nn.Linear(edge_feats, n_heads, bias=False) if edge_feats > 0 else None

        self.attn_drop   = nn.Dropout(attn_drop)
        self.edge_drop   = edge_drop
        self.leaky_relu  = nn.LeakyReLU(negative_slope, inplace=True)
        self.activation  = activation

        self.reset_parameters()

    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_normal_(self.src_fc.weight, gain=gain)
        if self.dst_fc is not None:
            nn.init.xavier_normal_(self.dst_fc.weight, gain=gain)
        nn.init.xavier_normal_(self.attn_src_fc.weight, gain=gain)
        if self.attn_dst_fc is not None:
            nn.init.xavier_normal_(self.attn_dst_fc.weight, gain=gain)
        if self.attn_edge_fc is not None:
            nn.init.xavier_normal_(self.attn_edge_fc.weight, gain=gain)

    def forward(self, graph, feat_src, feat_edge=None):
        with graph.local_scope():
            if graph.is_block:
                feat_dst = feat_src[: graph.number_of_dst_nodes()]
            else:
                feat_dst = feat_src

            if self._use_symmetric_norm:
                degs = graph.srcdata['deg']
                norm = torch.pow(degs, -0.5)
                shp  = norm.shape + (1,) * (feat_src.dim() - 1)
                feat_src = feat_src * norm.reshape(shp)

            feat_src_fc = self.src_fc(feat_src).view(-1, self._n_heads, self._out_feats)
            feat_dst_fc = self.dst_fc(feat_dst).view(-1, self._n_heads, self._out_feats)
            attn_src    = self.attn_src_fc(feat_src).view(-1, self._n_heads, 1)

            graph.srcdata.update({'feat_src_fc': feat_src_fc, 'attn_src': attn_src})

            if self.attn_dst_fc is not None:
                attn_dst = self.attn_dst_fc(feat_dst).view(-1, self._n_heads, 1)
                graph.dstdata.update({'attn_dst': attn_dst})
                graph.apply_edges(fn.u_add_v('attn_src', 'attn_dst', 'attn_node'))
            else:
                graph.apply_edges(fn.copy_u('attn_src', 'attn_node'))

            e = graph.edata['attn_node']
            if feat_edge is not None:
                attn_edge = self.attn_edge_fc(feat_edge).view(-1, self._n_heads, 1)
                graph.edata['attn_edge'] = attn_edge
                e = e + attn_edge
            e = self.leaky_relu(e)

            if self.training and self.edge_drop > 0:
                perm  = torch.randperm(graph.number_of_edges(), device=e.device)
                bound = int(graph.number_of_edges() * self.edge_drop)
                eids  = perm[bound:]
                graph.edata['a'] = torch.zeros_like(e)
                graph.edata['a'][eids] = self.attn_drop(edge_softmax(graph, e[eids], eids=eids))
            else:
                graph.edata['a'] = self.attn_drop(edge_softmax(graph, e))

            graph.update_all(fn.u_mul_e('feat_src_fc', 'a', 'm'), fn.sum('m', 'feat_src_fc'))
            rst = graph.dstdata['feat_src_fc']

            if self._use_symmetric_norm:
                degs = graph.dstdata['deg']
                norm = torch.pow(degs, 0.5)
                shp  = norm.shape + (1,) * feat_dst.dim()
                rst  = rst * norm.reshape(shp)

            # Residual
            if self.dst_fc is not None:
                rst = rst + feat_dst_fc
            else:
                rst = rst + self.bias

            if self.activation is not None:
                rst = self.activation(rst, inplace=True)

            return rst


class GNN(nn.Module):
    """Multi-layer GNN supporting GAT, GATE, SAGE, and GCN message passing."""

    def __init__(
        self,
        node_feats,
        edge_feats,
        n_classes,
        n_layers,
        n_heads,
        n_hidden,
        edge_emb,
        activation,
        dropout,
        input_drop,
        attn_drop,
        edge_drop,
        use_attn_dst=True,
        allow_zero_in_degree=False,
        mpnn='gat',
        jk=False,
    ):
        super().__init__()
        self.n_layers  = n_layers
        self.n_heads   = n_heads
        self.n_hidden  = n_hidden
        self.n_classes = n_classes
        self.mpnn      = mpnn
        self.jk        = jk

        self.convs        = nn.ModuleList()
        self.norms        = nn.ModuleList()
        self.node_encoder = nn.Linear(node_feats, n_hidden)

        if edge_emb > 0:
            self.edge_encoder = nn.ModuleList()
        else:
            self.edge_encoder = None

        for i in range(n_layers):
            in_hidden  = n_heads * n_hidden if i > 0 else n_hidden
            out_hidden = n_heads * n_hidden

            if self.edge_encoder is not None:
                self.edge_encoder.append(nn.Linear(edge_feats, edge_emb))

            if mpnn in ('gat', 'gate'):
                self.convs.append(
                    GATConv(
                        in_hidden,
                        edge_emb,
                        n_hidden,
                        n_heads=n_heads,
                        attn_drop=attn_drop,
                        edge_drop=edge_drop,
                        use_attn_dst=use_attn_dst,
                        allow_zero_in_degree=allow_zero_in_degree,
                        use_symmetric_norm=False,
                    )
                )
            elif mpnn == 'sage':
                self.convs.append(SAGEConv(in_hidden, out_hidden, aggregator_type='mean'))
            else:  # gcn
                self.convs.append(GraphConv(in_hidden, out_hidden))

            self.norms.append(nn.BatchNorm1d(out_hidden))

        self.pred_linear = nn.Linear(n_heads * n_hidden, n_classes)
        self.input_drop  = nn.Dropout(input_drop)
        self.dropout     = nn.Dropout(dropout)
        self.activation  = activation

    def forward(self, g):
        subgraphs = g if isinstance(g, list) else [g] * self.n_layers

        h = subgraphs[0].srcdata['feat']
        h = self.node_encoder(h)
        h = F.relu(h, inplace=True)
        h = self.input_drop(h)

        h_local = []
        h_last  = None

        for i in range(self.n_layers):
            efeat_emb = None
            if self.mpnn == 'gate' and self.edge_encoder is not None:
                efeat     = subgraphs[i].edata['feat']
                efeat_emb = F.relu(self.edge_encoder[i](efeat), inplace=True)

            h = self.convs[i](subgraphs[i], h, efeat_emb).flatten(1, -1)

            if h_last is not None:
                h = h + h_last[: h.shape[0], :]
            h_last = h

            h = self.norms[i](h)
            h = self.activation(h, inplace=True)
            h = self.dropout(h)
            h_local.append(h)

        if self.jk:
            h_local = [t[: h.shape[0], :] for t in h_local]
            h = torch.sum(torch.stack(h_local), dim=0)

        return self.pred_linear(h)


print('Model classes defined.')

## 5. Utility Functions

In [ ]:
def set_seed(seed_val=0):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    dgl.random.seed(seed_val)


def load_data(dataset_name):
    data       = DglNodePropPredDataset(name=dataset_name, root='data/ogb')
    evaluator  = Evaluator(name=dataset_name)
    split_idx  = data.get_idx_split()
    train_idx  = split_idx['train']
    val_idx    = split_idx['valid']
    test_idx   = split_idx['test']
    graph, labels = data[0]
    graph.ndata['labels'] = labels
    return graph, labels, train_idx, val_idx, test_idx, evaluator


def preprocess(graph, labels, train_idx):
    global N_NODE_FEATS
    # Aggregate edge features to node features via sum
    graph.update_all(fn.copy_e('feat', 'feat_copy'), fn.sum('feat_copy', 'feat'))
    N_NODE_FEATS = graph.ndata['feat'].shape[-1]

    # Training labels as additional input features (others stay zero)
    graph.ndata['train_labels_onehot'] = torch.zeros(graph.number_of_nodes(), N_CLASSES)
    graph.ndata['train_labels_onehot'][train_idx, labels[train_idx, 0]] = 1
    graph.ndata['deg'] = graph.out_degrees().float().clamp(min=1)

    graph.create_formats_()
    return graph, labels


def gen_model():
    n_feats = (N_NODE_FEATS + N_CLASSES) if USE_LABELS else N_NODE_FEATS
    return GNN(
        n_feats,
        N_EDGE_FEATS,
        N_CLASSES,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        n_hidden=N_HIDDEN,
        edge_emb=16,
        activation=F.relu,
        dropout=DROPOUT,
        input_drop=INPUT_DROP,
        attn_drop=ATTN_DROP,
        edge_drop=EDGE_DROP,
        use_attn_dst=not NO_ATTN_DST,
        mpnn=MPNN,
        jk=JK,
    )


def add_labels(graph, idx):
    feat = graph.srcdata['feat']
    labels_onehot = torch.zeros([feat.shape[0], N_CLASSES], device=device)
    labels_onehot[idx] = graph.srcdata['train_labels_onehot'][idx]
    graph.srcdata['feat'] = torch.cat([feat, labels_onehot], dim=-1)


print('Utility functions defined.')

## 6. Training and Evaluation

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    loss_sum, total = 0, 0

    for input_nodes, output_nodes, subgraphs in dataloader:
        subgraphs       = [b.to(device) for b in subgraphs]
        new_train_idx   = torch.arange(len(output_nodes), device=device)

        if USE_LABELS:
            train_labels_idx = torch.arange(len(output_nodes), len(input_nodes), device=device)
            add_labels(subgraphs[0], train_labels_idx)

        pred  = model(subgraphs)
        loss  = criterion(pred[new_train_idx],
                          subgraphs[-1].dstdata['labels'][new_train_idx].float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * len(new_train_idx)
        total    += len(new_train_idx)

    return loss_sum / total


@torch.no_grad()
def evaluate(model, dataloader, labels, train_idx, val_idx, test_idx, criterion, evaluator):
    model.eval()
    preds      = torch.zeros(labels.shape).to(device)
    eval_times = 1

    for _ in range(eval_times):
        for input_nodes, output_nodes, subgraphs in dataloader:
            subgraphs = [b.to(device) for b in subgraphs]
            if USE_LABELS:
                all_idx = list(range(len(input_nodes)))
                add_labels(subgraphs[0], all_idx)
            pred = model(subgraphs)
            preds[output_nodes] += pred

    preds /= eval_times

    train_loss = criterion(preds[train_idx], labels[train_idx].float()).item()
    val_loss   = criterion(preds[val_idx],   labels[val_idx].float()).item()
    test_loss  = criterion(preds[test_idx],  labels[test_idx].float()).item()

    return (
        evaluator(preds[train_idx], labels[train_idx]),
        evaluator(preds[val_idx],   labels[val_idx]),
        evaluator(preds[test_idx],  labels[test_idx]),
        train_loss, val_loss, test_loss,
        preds,
    )


print('Training/evaluation functions defined.')

## 7. Main Run Function

In [ ]:
def run(graph, labels, train_idx, val_idx, test_idx, evaluator, n_running):
    evaluator_wrapper = lambda pred, lbls: evaluator.eval(
        {'y_pred': pred, 'y_true': lbls}
    )['rocauc']

    train_batch_size = (len(train_idx) + 9) // 10

    train_sampler  = MultiLayerNeighborSampler([32] * N_LAYERS)
    train_dataloader = DataLoader(
        graph=graph.cpu(),
        indices=train_idx.cpu(),
        graph_sampler=train_sampler,
        batch_size=train_batch_size,
        shuffle=True,
        num_workers=4,
    )

    eval_sampler  = MultiLayerNeighborSampler([100] * N_LAYERS)
    eval_dataloader = DataLoader(
        graph=graph.cpu(),
        indices=torch.cat([train_idx.cpu(), val_idx.cpu(), test_idx.cpu()]),
        graph_sampler=eval_sampler,
        batch_size=65536,
        shuffle=False,
        num_workers=4,
    )

    criterion  = nn.BCEWithLogitsLoss()
    model      = gen_model().to(device)
    optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.75, patience=50, verbose=True
    )

    total_time = 0
    best_val_score, final_test_score = 0, 0
    val_score  = 0
    final_pred = None

    for epoch in range(1, N_EPOCHS + 1):
        tic  = time.time()
        loss = train_epoch(model, train_dataloader, criterion, optimizer)
        toc  = time.time()
        total_time += toc - tic

        if epoch == N_EPOCHS or epoch % EVAL_EVERY == 0 or epoch % LOG_EVERY == 0:
            train_score, val_score, test_score, train_loss, val_loss, test_loss, pred = evaluate(
                model, eval_dataloader, labels, train_idx, val_idx, test_idx, criterion, evaluator_wrapper
            )

            if val_score > best_val_score:
                best_val_score   = val_score
                final_test_score = test_score
                final_pred       = pred

            if epoch % LOG_EVERY == 0:
                print(
                    f'Epoch: {epoch:04d} | '
                    f'Loss: {loss:.4f} | '
                    f'Train: {100 * train_score:.2f}% | '
                    f'Valid: {100 * val_score:.2f}% | '
                    f'Test: {100 * test_score:.2f}% | '
                    f'Best Valid: {100 * best_val_score:.2f}% | '
                    f'Best Test: {100 * final_test_score:.2f}%'
                )

        lr_scheduler.step(val_score)

    if SAVE_PRED and final_pred is not None:
        os.makedirs('./output', exist_ok=True)
        torch.save(F.softmax(final_pred, dim=1), f'./output/{n_running}.pt')

    return best_val_score, final_test_score


print('Run function defined.')

## 8. Load and Preprocess Data

In [ ]:
print('Loading data ...')
graph, labels, train_idx, val_idx, test_idx, evaluator = load_data(DATASET_NAME)

print('Preprocessing ...')
graph, labels = preprocess(graph, labels, train_idx)

labels, train_idx, val_idx, test_idx = (
    labels.to(device),
    train_idx.to(device),
    val_idx.to(device),
    test_idx.to(device),
)

print(f'Graph:       {graph}')
print(f'Labels:      {labels.shape}')
print(f'Train nodes: {len(train_idx)}')
print(f'Val nodes:   {len(val_idx)}')
print(f'Test nodes:  {len(test_idx)}')
print(f'Node features (after preprocess): {N_NODE_FEATS}')

## 9. Train the Model

In [ ]:
all_val_scores  = []
all_test_scores = []

for i in range(N_RUNS):
    print(f'\n=== Run {i + 1} / {N_RUNS} ===')
    set_seed(SEED + i)
    val_score, test_score = run(
        graph, labels, train_idx, val_idx, test_idx, evaluator, n_running=i + 1
    )
    all_val_scores.append(val_score)
    all_test_scores.append(test_score)
    print(f'Run {i + 1} finished – Val ROC-AUC: {val_score:.4f} | Test ROC-AUC: {test_score:.4f}')

print('\n=== Summary ===')
print(f'Val  ROC-AUC: {np.mean(all_val_scores):.4f} ± {np.std(all_val_scores):.4f}')
print(f'Test ROC-AUC: {np.mean(all_test_scores):.4f} ± {np.std(all_test_scores):.4f}')